In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.aml_lakehouse.bronze_transactions")

silver = (
    bronze
    .withColumn("txn_ts", F.to_timestamp("timestamp", "yyyy/MM/dd HH:mm"))
    .withColumn("from_account_id", F.concat_ws("_", "from_bank", "from_account"))
    .withColumn("to_account_id",   F.concat_ws("_", "to_bank", "to_account"))
    .drop("timestamp")
)

# Exact duplicates on business columns (audit columns excluded)
business_cols = [c for c in silver.columns if not c.startswith("_")]
silver_dedup = silver.dropDuplicates(business_cols)
print("rows before:", silver.count(), "| after dedup:", silver_dedup.count())

# Stable transaction id from the business columns
silver_final = silver_dedup.withColumn(
    "transaction_id", F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in business_cols]), 256)
)

silver_final.write.format("delta").mode("overwrite").saveAsTable("workspace.aml_lakehouse.silver_transactions")

# Accounts dimension: every account seen as sender or receiver
accounts = (
    silver_final.select(F.col("from_account_id").alias("account_id"),
                        F.col("from_bank").alias("bank"), F.col("from_account").alias("account"))
    .union(
        silver_final.select(F.col("to_account_id").alias("account_id"),
                            F.col("to_bank").alias("bank"), F.col("to_account").alias("account")))
    .distinct()
)
accounts.write.format("delta").mode("overwrite").saveAsTable("workspace.aml_lakehouse.silver_accounts")
print("distinct accounts:", accounts.count())

In [0]:
s = spark.table("workspace.aml_lakehouse.silver_transactions")
display(s.select(
    F.sum(F.col("txn_ts").isNull().cast("int")).alias("null_ts"),
    F.sum((F.col("amount_paid") <= 0).cast("int")).alias("non_positive_paid"),
    F.sum(F.col("from_account_id").isNull().cast("int")).alias("null_from"),
    F.countDistinct("transaction_id").alias("distinct_txn_ids"),
    F.count("*").alias("total_rows"),
))

In [0]:
s = spark.table("workspace.aml_lakehouse.silver_transactions")
display(s.groupBy("is_laundering").count())